In [ ]:
!pip install openai pydantic pandas

In [ ]:
from typing import List, Optional
from pydantic import BaseModel, Field


class InvoiceItem(BaseModel):
    name: str = Field(description="Повна назва знайденого товару/послуги")
    quantity: float = Field(description="Кількість куплених одиниць", default=1.0)
    price: Optional[float] = Field(description="Ціна за одиницю товару", default=None)
    total: Optional[float] = Field(description="Підсумкова вартість даної товарної позиції", default=None)


class ExtractedInvoice(BaseModel):
    reasoning: str = Field(
        description=(
            "Chain-of-Thought міркування: 1) виправлення помилок OCR у назві закладу; "
            "2) порівняння з токенними сутностями Лаб. 5; 3) виявлення та групування товарів; "
            "4) перевірка математичної суми позицій відносно підсумкового TOTAL."
        )
    )
    merchant_name: Optional[str] = Field(description="Назва торгової точки або мережі", default=None)
    date: Optional[str] = Field(description="Дата транзакції у форматі YYYY-MM-DD", default=None)
    items: List[InvoiceItem] = Field(description="Список розпізнаних позицій", default_factory=list)
    total_amount: Optional[float] = Field(description="Підсумкова сума до сплати (TOTAL)", default=None)


print("[+] Схему ExtractedInvoice скомпільовано.")

[+] Схему ExtractedInvoice скомпільовано.


In [ ]:
import json
import os
import pandas as pd
from google.colab import userdata
from openai import OpenAI

# 1. Ініціалізація клієнта Groq
raw_key = userdata.get("groq-api-key")
if not raw_key:
    raise ValueError("Секрет 'groq-api-key' не знайдено! Перевір Secrets у Google Colab.")

client_groq = OpenAI(
    api_key=raw_key.strip(),
    base_url="https://api.groq.com/openai/v1",
)
print("[+] Клієнт Groq ініціалізовано.")

# Перевіримо список доступних моделей, щоб підібрати актуальну для Кроку 3 і 4
try:
    available_models = [m.id for m in client_groq.models.list().data]
    print("[i] Доступні моделі на Groq:", available_models[:10])
except Exception as e:
    print(f"[!] Не вдалося отримати список моделей: {e}")

# 2. Зчитування тексту з Лабораторної № 4 (ocr_result_degraded.txt)
ocr_filename = "ocr_result_degraded.txt"
try:
    with open(ocr_filename, "r", encoding="utf-8") as f:
        raw_ocr_text = f.read().strip()
    print(f"[+] Артефакт Лаб. 4 ({ocr_filename}) завантажено (довжина: {len(raw_ocr_text)} символів).")
except FileNotFoundError:
    raw_ocr_text = ""
    print(f"[!] '{ocr_filename}' не знайдено.")

# 3. Зчитування артефакту токенного NER з Лабораторної № 5 (ner_extracted.json)
ner_filename = "ner_extracted.json"
try:
    with open(ner_filename, "r", encoding="utf-8") as f:
        bert_entities = json.load(f)
    print(f"[+] Артефакт Лаб. 5 ({ner_filename}) завантажено.")
except FileNotFoundError:
    bert_entities = {}
    print(f"[!] '{ner_filename}' не знайдено.")

print("\n--- Сутності, розпізнані моделлю BERT у Лабораторній № 5 ---")
print(json.dumps(bert_entities, ensure_ascii=False, indent=2))

[+] Клієнт Groq ініціалізовано.
[i] Доступні моделі на Groq: ['meta-llama/llama-prompt-guard-2-22m', 'openai/gpt-oss-120b', 'canopylabs/orpheus-arabic-saudi', 'allam-2-7b', 'qwen/qwen3.8-27b', 'openai/gpt-oss-20b', 'openai/gpt-oss-safeguard-20b', 'canopylabs/orpheus-v1-english', 'whisper-large-v3-turbo', 'whisper-large-v3']
[+] Артефакт Лаб. 4 (ocr_result_degraded.txt) завантажено (довжина: 466 символів).
[+] Артефакт Лаб. 5 (ner_extracted.json) завантажено.

--- Сутності, розпізнані моделлю BERT у Лабораторній № 5 ---
{
  "variant": 6,
  "source_file": "ocr_result_clean.txt",
  "model_name": "Babelscape/wikineural-multilingual-ner",
  "aggregation_strategy": "simple",
  "entities_count": 6,
  "entities": [
    {
      "entity_group": "LOC",
      "word": "Кобилян",
      "score": 0.6734,
      "start": 24,
      "end": 31
    },
    {
      "entity_group": "MISC",
      "word": "Супермар",
      "score": 0.4897,
      "start": 72,
      "end": 80
    },
    {
      "entity_group": "MI

In [ ]:
# Підстав модель із тих, що повернув client_groq.models.list() на Кроці 2
# (наприклад, "qwen/qwen3.8-27b", "llama-3.3-70b-versatile", "qwen/qwen3-32b" тощо)
MODEL_NAME = "qwen/qwen3.8-27b"

print("[*] Тест генерації: Temperature = 1.0 (Без обмеження схеми)")

response_chaos = client_groq.chat.completions.create(
    model=MODEL_NAME,
    temperature=1.0,
    messages=[
        {"role": "system", "content": "Витягни дані з цього чека у JSON."},
        {"role": "user", "content": raw_ocr_text},
    ],
)

chaos_output = response_chaos.choices[0].message.content
print("\n--- Результат моделі при Т=1.0 ---")
print(chaos_output)

[*] Тест генерації: Temperature = 1.0 (Без обмеження схеми)

--- Результат моделі при Т=1.0 ---
```json
{
  "seller_info": {
    "name": "ІРАХП-ФРАНКІВСЬК",
    "address": "РЫЛ, КОБИЛЯНСЬКІІ 56",
    "pin": "325497419188"
  },
  "transaction_info": {
    "card_type": "KYCLDAK",
    "barcode": "4820242913786",
    "card_number_masked": "4820242913786",
    "transaction_id": "1049932",
    "method": "БЕЗГОТІВКІВА",
    "bank": "ПРМВАТБАНК",
    "card_bin": "STIF08Q4",
    "authorization_code": "118193519777",
    "merchant_code": "121277",
    "terminal_id": "3101927131"
  },
  "items": [
    {
      "sku": "1049932",
      "name": "Eskaro Стрічка нальнона",
      "category_code": "DEP OL ЕКСПЕРТ 48нН",
      "barcodes": [
        {
          "value": "4820242913786",
          "prefix": "Штрих код"
        }
      ],
      "weight": "",
      "unit_price": 351.14,
      "price": 129.01,
      "quantity": "X",
      "note": "ДЕП OL ЕКСПЕРТ 48нН"
    },
    {
      "sku": "654422",
      

In [ ]:
import json
import re

# 1. Експортуємо схему в JSON Schema
schema = ExtractedInvoice.model_json_schema()

# 2. Формуємо системну інструкцію
system_prompt = (
    "Ти — експертний автономний модуль інтелектуального вилучення даних з документів. "
    "Тобі надано: 1) сирий текст чека після OCR (Лаб. 4); 2) попередньо розпізнані моделлю BERT сутності (Лаб. 5). "
    "Твоє завдання — зіставити ці джерела, виправити шуми розпізнавання, згрупувати товари з їхніми цінами "
    "та повернути СТРОГО валідний JSON за наданою схемою.\n"
    f"Контракт схеми: {json.dumps(schema, ensure_ascii=False)}\n"
    "ВИМОГИ:\n"
    "- Поверни ТІЛЬКИ чистий JSON-об'єкт без маркдаун-тегів.\n"
    "- Обов'язково заповни поле reasoning, де опиши хід думок та звірку цін.\n"
    "- Використовуй точні імена ключів: reasoning, merchant_name, date, items, total_amount."
)

# Приклад шаблону (Few-Shot)
few_shot_example = (
    "\n\nПРИКЛАД ОБРОБКИ:\n"
    "Текст: 'ATB-Markt 12.05.2025 HLIB 24.00 MOLOKO 36.00 SUMA: 60.00'\n"
    "Відповідь: {\n"
    '  "reasoning": "Виправлено назву на ATB-Market; дата 2025-05-12; сума позицій 24+36=60 відповідає підсумку 60.",\n'
    '  "merchant_name": "ATB-Market",\n'
    '  "date": "2025-05-12",\n'
    '  "items": [\n'
    '    {"name": "HLIB", "quantity": 1.0, "price": 24.0, "total": 24.0},\n'
    '    {"name": "MOLOKO", "quantity": 1.0, "price": 36.0, "total": 36.0}\n'
    "  ],\n"
    '  "total_amount": 60.0\n'
    "}"
)

system_prompt = system_prompt + few_shot_example

# 3. Виклик Groq API (використовуємо MODEL_NAME з Кроку 2/3, або вкажи потрібну модель явно)
active_model = MODEL_NAME if "MODEL_NAME" in globals() else "qwen/qwen3.8-27b"

response = client_groq.chat.completions.create(
    model=active_model,
    temperature=0.0,
    messages=[
        {"role": "system", "content": system_prompt},
        {
            "role": "user",
            "content": (
                f"=== ДЖЕРЕЛО 1: Сирий текст з Лабораторної № 4 (OCR) ===\n{raw_ocr_text}\n\n"
                f"=== ДЖЕРЕЛО 2: Сутності BERT з Лабораторної № 5 ===\n{json.dumps(bert_entities, ensure_ascii=False)}"
            ),
        },
    ],
    response_format={"type": "json_object"},
)

raw_json = (response.choices[0].message.content or "").strip()
# На випадок якщо модель все ж додала ```json ... ```
cleaned_json = re.sub(r"^```(?:json)?\s*|\s*```$", "", raw_json, flags=re.DOTALL).strip()

# 4. Валідація контракту через Pydantic
llm_data = ExtractedInvoice.model_validate_json(cleaned_json)

print("=================== ХІД ДУМОК МОДЕЛІ (Chain-of-Thought) ===================")
print(llm_data.reasoning)
print("==========================================================================")
print(f"Магазин: {llm_data.merchant_name}")
print(f"Дата: {llm_data.date}")
print(f"Загальна сума: {llm_data.total_amount}")
print("\nЗнайдені позиції:")
for it in llm_data.items:
    print(f" - {it.name:<25} | К-сть: {it.quantity:<5} | Ціна: {it.price} | Разом: {it.total}")

=================== ХІД ДУМОК МОДЕЛІ (Chain-of-Thought) ===================
1) Виправлення назви закладу: OCR 'ІRAHП-ФРАНКІВСЬК' та BERT 'Супермар' вказують на 'Супермаркет ФРАНКІВСЬК' (або аналогічну мережу, напр. 'Сільпо'/'АТБ', але за текстом 'ІRAHП' схоже на 'СІЛЬПО' або 'ФРАНКІВСЬК' як бренд). Враховуючи адресу 'КОБИЛЯНСЬКІі 56' (вул. Кобилянського), це типовий формат чека. Назву закладу інтерпретуємо як 'Супермаркет ФРАНКІВСЬК' або просто 'ФРАНКІВСЬК' з урахуванням шуму. 2) Порівняння з BERT: BERT знайшов 'Eskar' (OCR: 'Eskaro') та 'Kirchland PrimeTex' (OCR: 'Kirch |and PrifeTex 1uf'). 3) Групування товарів: 
- Позиція 1: 'Eskaro Стрічка налнона' (OCR: 'Eskaro Стрічка налнона', BERT: 'Eskar'). Ціна: 351.14. Кількість: 1. 
- Позиція 2: 'GRIVER Герне' (OCR: 'GRIVER Герне'). Ціна: 129.01 (OCR: 'X 129,01'). Кількість: 1. 
- Позиція 3: 'Kirchland PrimeTex 1uf Грун' (OCR: 'Kirch |and PrifeTex 1uf Грун', BERT: 'Kirchland PrimeTex'). Ціна: 1433.81 (OCR: '1433.81'). Кількість: 1. 
4) Пере

In [ ]:
import pandas as pd

print("\n[*] ПОРІВНЯЛЬНИЙ АНАЛІЗ: BERT-NER (Лаб. 5) vs LLM Structured Outputs (Лаб. 6)\n")

# Підтримуємо обидва формати bert_entities: dict з "entities" або плоский dict сутностей
if isinstance(bert_entities, dict) and isinstance(bert_entities.get("entities"), list):
    raw_ents = bert_entities["entities"]
    bert_model_label = bert_entities.get("model_name", "BERT-NER")
else:
    raw_ents = []
    bert_model_label = "BERT-NER"

grouped_ents = {}
for ent in raw_ents:
    if isinstance(ent, dict):
        grp = ent.get("entity_group", "OTHER")
        word = str(ent.get("word", "")).strip()
        score = ent.get("score")
        if word:
            formatted = f"{word} ({score:.2f})" if isinstance(score, (int, float)) else word
            grouped_ents.setdefault(grp, []).append(formatted)

loc_str = ", ".join(grouped_ents.get("LOC", [])) or "Не знайдено"
misc_str = ", ".join(grouped_ents.get("MISC", [])) or "Не знайдено"
org_str = ", ".join(grouped_ents.get("ORG", [])) or "Не знайдено"
per_str = ", ".join(grouped_ents.get("PER", [])) or "Не знайдено"

bert_store_summary = (
    f"ORG: {org_str} | LOC: {loc_str} | MISC: {misc_str}"
    if any(grouped_ents.get(k) for k in ("ORG", "LOC", "MISC"))
    else "Не знайдено"
)

llm_items_summary = (
    ", ".join(
        f"{it.name} (qty={it.quantity}, price={it.price}, total={it.total})"
        for it in llm_data.items
    )
    if llm_data.items
    else "Не знайдено"
)

active_llm_name = MODEL_NAME if "MODEL_NAME" in globals() else "LLM Groq"

comparison_data = [
    {
        "Критерій / Поле": "Назва торгової точки / локація",
        "BERT-NER (Лабораторна № 5)": bert_store_summary,
        "LLM Groq (Лабораторна № 6)": llm_data.merchant_name or "Не знайдено",
        "Коментар щодо стійкості до шуму": (
            f"{bert_model_label} виділяє фрагменти ('Кобилян', 'Супермар', 'Kirchland PrimeTex'); "
            "LLM відновлює цілісну назву закладу за контекстом чека"
        ),
    },
    {
        "Критерій / Поле": "Формат дати",
        "BERT-NER (Лабораторна № 5)": "Окремий клас DATE відсутній у виводі NER",
        "LLM Groq (Лабораторна № 6)": llm_data.date or "Не знайдено",
        "Коментар щодо стійкості до шуму": "BERT не розмічає дати в цій конфігурації; LLM знаходить і нормалізує у YYYY-MM-DD",
    },
    {
        "Критерій / Поле": "Структуризація товарів",
        "BERT-NER (Лабораторна № 5)": f"Ізольовані сутності ({len(raw_ents)} шт., PER: {per_str}) без цін і кількості",
        "LLM Groq (Лабораторна № 6)": f"Масив з {len(llm_data.items)} позицій: {llm_items_summary}",
        "Коментар щодо стійкості до шуму": "BERT дає лише розрізнені мітки; LLM формує зв'язані об'єкти (Name + Qty + Price + Total)",
    },
    {
        "Критерій / Поле": "Підсумкова сума (TOTAL)",
        "BERT-NER (Лабораторна № 5)": "Не виділена (фрагмент 'Су' з низьким score 0.38)",
        "LLM Groq (Лабораторна № 6)": str(llm_data.total_amount) if llm_data.total_amount is not None else "Не знайдено",
        "Коментар щодо стійкості до шуму": "LLM витягує підсумок і дозволяє звірити його з сумою всіх позицій",
    },
]

df_benchmark = pd.DataFrame(comparison_data)
pd.set_option("display.max_colwidth", None)
display(df_benchmark)


[*] ПОРІВНЯЛЬНИЙ АНАЛІЗ: BERT-NER (Лаб. 5) vs LLM Structured Outputs (Лаб. 6)



,Критерій / Поле,BERT-NER (Лабораторна № 5),LLM Groq (Лабораторна № 6),Коментар щодо стійкості до шуму
0,Назва торгової точки / локація,"ORG: Не знайдено | LOC: Кобилян (0.67) | MISC: Супермар (0.49), Касир (0.45), Eskar (0.44), Kirchland PrimeTex (0.69), Су (0.38)",Супермаркет ФРАНКІВСЬК,"Babelscape/wikineural-multilingual-ner виділяє фрагменти ('Кобилян', 'Супермар', 'Kirchland PrimeTex'); LLM відновлює цілісну назву закладу за контекстом чека"
1,Формат дати,Окремий клас DATE відсутній у виводі NER,Не знайдено,BERT не розмічає дати в цій конфігурації; LLM знаходить і нормалізує у YYYY-MM-DD
2,Структуризація товарів,"Ізольовані сутності (6 шт., PER: Не знайдено) без цін і кількості","Масив з 3 позицій: Eskaro Стрічка нальнона (qty=1.0, price=351.14, total=351.14), GRIVER Герне (qty=1.0, price=129.01, total=129.01), Kirchland PrimeTex 1uf Грун (qty=1.0, price=1433.81, total=1433.81)",BERT дає лише розрізнені мітки; LLM формує зв'язані об'єкти (Name + Qty + Price + Total)
3,Підсумкова сума (TOTAL),Не виділена (фрагмент 'Су' з низьким score 0.38),1913.96,LLM витягує підсумок і дозволяє звірити його з сумою всіх позицій


In [ ]:
print("\n[*] Детермінована перевірка арифметичного балансу (Cross-Validation)")

# Рахуємо суму позицій; якщо у позиції немає total, але є quantity і price — підрахуємо на льоту
item_totals = []
missing_totals_count = 0

for it in llm_data.items:
    if it.total is not None:
        item_totals.append(float(it.total))
    elif it.price is not None and it.quantity is not None:
        item_totals.append(round(float(it.quantity) * float(it.price), 2))
    else:
        missing_totals_count += 1

calc_sum = round(sum(item_totals), 2)
doc_total = float(llm_data.total_amount) if llm_data.total_amount is not None else 0.0
discrepancy = round(abs(calc_sum - doc_total), 2)

print(f"Розрахована сума окремих позицій: {calc_sum:.2f}")
print(f"Сума, заявлена у чеку (TOTAL): {doc_total:.2f}")
print(f"Абсолютна розбіжність: {discrepancy:.2f}")
if missing_totals_count > 0:
    print(f"[i] Позицій без числової суми після OCR: {missing_totals_count}")

# Допустима похибка до 0.05 (на округлення податку або копійок)
is_valid = bool(doc_total > 0 and discrepancy <= 0.05 and llm_data.merchant_name)

if is_valid:
    print("\n[✓] СТАТУС: VALIDATION PASSED. Транзакція консистентна -> Маршрутизація в ERP/Таблицю.")
else:
    print("\n[!] СТАТУС: VALIDATION FAILED. Баланс не зійшовся -> Маршрутизація оператору (Human-in-the-Loop).")


[*] Детермінована перевірка арифметичного балансу (Cross-Validation)
Розрахована сума окремих позицій: 1913.96
Сума, заявлена у чеку (TOTAL): 1913.96
Абсолютна розбіжність: 0.00

[✓] СТАТУС: VALIDATION PASSED. Транзакція консистентна -> Маршрутизація в ERP/Таблицю.
